# HAMIC · TPU Pretraining

Reproducible, non-interactive TPU training and benchmarking notebook for **HAMIC-LM** (`hamic_jax`).

- Edit parameters directly in **Cell 1** and use **Run all** (Google Colab TPU v5e-1) or **Save & Run All** (Kaggle TPU v5e-8).
- `MODE = "bench"` runs a short local throughput and loss check without publishing to Hugging Face Hub.
- `MODE = "train"` runs checkpoint-resumable training with checkpoint-gated shard window streaming (`SHARD_WINDOW > 0`) and automatic Hub resume.

In [ ]:
HAMIC_SOURCE = "git+https://github.com/minhhhduc/HAMIC-LM@main"
HUB_REPO = "HaianCao/hamic-runs"
DATA_REPO = "HaianCao/hamic-data"
XET_HIGH_PERFORMANCE = True

CONFIG_PATH = "configs/train/es/hamic-es-v014.yaml"
RUN_ID = ""          # Leave empty to use "<config.run_id>-tpu"
MODE = "bench"       # "bench" (un-published sanity check) or "train" (Hub-resumed run)
SHARD_WINDOW = 8     # 0 = download full corpus up front; >0 = stream N shards per window
RECOMPUTE = True     # Gradient checkpointing across transformer layers
SESSION_HOURS = 8.5  # Wall-clock session budget when MODE == "train"
SHOW_PROGRESS = True # Plain-text tqdm progress bars for train and eval
KEEP_LAST_CHECKPOINTS = 2  # Remote Hub checkpoints kept per run (0 = keep all)

# Benchmark overrides (used only when MODE == "bench")
BENCH_STEPS = 10
BENCH_MICRO_BATCH = 0   # 0 = auto (16 on 1-device TPU v5e-1, else config.micro_batch_size)
BENCH_SHARD_WINDOW = 2
BENCH_LR = None
BENCH_WARMUP = None

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

if XET_HIGH_PERFORMANCE:
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

work_root = Path("/content/hamic-tpu") if Path("/content").is_dir() else Path(tempfile.gettempdir()) / "hamic-tpu"
work_root.mkdir(parents=True, exist_ok=True)
hf_cache_root = work_root / "hf_cache"
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_XET_CACHE"] = str(hf_cache_root / "xet")
source_dir = work_root / "HAMIC-LM"

src_spec = HAMIC_SOURCE[4:] if HAMIC_SOURCE.startswith("git+") else HAMIC_SOURCE
repo_url, _, branch = src_spec.partition("@")
branch = branch or "main"

if not (source_dir / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(source_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(source_dir), "fetch", "--depth", "1", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(source_dir), "checkout", "-B", branch, f"origin/{branch}"], check=True)

installed_commit = subprocess.check_output(["git", "-C", str(source_dir), "rev-parse", "HEAD"], text=True).strip()
src_path = str(source_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from hamic_jax.dependencies import ensure_dependencies

installed_pkgs = ensure_dependencies()
if installed_pkgs:
    print("Installed packages:", ", ".join(installed_pkgs))

import hamic_jax  # noqa: F401

os.chdir(source_dir)
hamic_ready = installed_commit
run_configs = sorted(p.relative_to(source_dir).as_posix() for p in (source_dir / "configs" / "train").rglob("*.yaml"))
print(f"Source ready: {installed_commit[:12]} ({branch}) | work_root={work_root}")
print("Available training configs:")
for cfg in run_configs:
    print(f"  - {cfg}")

In [ ]:
import jax

assert hamic_ready == installed_commit, f"Source mismatch: {hamic_ready} != {installed_commit}"
assert hasattr(jax.lax, "pcast"), "jax.lax.pcast missing; upgrade JAX for TPU v5e collective support"
backend = jax.default_backend()
assert backend == "tpu", f"Expected TPU backend, got {backend!r}"

cache_dir = work_root / "jax_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
jax.config.update("jax_compilation_cache_dir", str(cache_dir))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)

devices = jax.devices()
tpu_kind = devices[0].device_kind if devices else "unknown"
print(f"Backend: {backend} ({tpu_kind}) | devices={len(devices)} | jax={jax.__version__} | commit={installed_commit[:12]}")

In [ ]:
from hamic_jax.checkpoint import HuggingFaceObjectStore

hub_token = ""
try:
    from google.colab import userdata
    hub_token = (userdata.get("HF_KEY") or "").strip()
except Exception:
    pass

if not hub_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hub_token = (UserSecretsClient().get_secret("HF_KEY") or "").strip()
    except Exception:
        pass

if not hub_token:
    hub_token = os.environ.get("HF_KEY", "").strip()

if not hub_token:
    raise RuntimeError("HF_KEY secret or environment variable is required.")

os.environ["HF_KEY"] = hub_token

run_store = HuggingFaceObjectStore(HUB_REPO, repo_type="model", private=True, create=True, token=hub_token)
data_store = HuggingFaceObjectStore(DATA_REPO, repo_type="dataset", private=True, create=False, token=hub_token)

account = run_store.whoami()
corpus_files: dict[str, int] = {}
for rel in data_store.list_files("tokenized/"):
    if not rel.endswith(".npy"):
        continue
    parts = Path(rel).parts
    if len(parts) >= 3:
        pref = "/".join(parts[:3])
        corpus_files[pref] = corpus_files.get(pref, 0) + 1

print(f"HF account: {account} | runs={HUB_REPO} | data={DATA_REPO}")
print("Available tokenized corpora (.npy shards):")
for pref, count in sorted(corpus_files.items()):
    print(f"  - {pref}: {count} shards")

In [ ]:
from dataclasses import replace
import pandas as pd
from IPython.display import display
from hamic_jax.data import fetch_corpus_metadata, fetch_tokenized, tokenized_prefix
from hamic_jax.presets import load_training_config
from hamic_jax.runs import run_prefix
from hamic_jax.spec import read_yaml

cfg_file = source_dir / CONFIG_PATH
if not cfg_file.is_file():
    raise FileNotFoundError(f"CONFIG_PATH not found: {CONFIG_PATH}. Choose from run_configs.")

config, preset = load_training_config(cfg_file)
raw_config = read_yaml(cfg_file)

run_id = RUN_ID.strip() or f"{config.run_id}-tpu"
_ = run_prefix(run_id)

corpus_pref = tokenized_prefix(config.tokenized)
n_shards = corpus_files.get(corpus_pref, 0)
if n_shards == 0:
    raise RuntimeError(f"No .npy shards found in {DATA_REPO} under prefix {corpus_pref!r}")

effective_window = int(SHARD_WINDOW)
if effective_window == 0 and n_shards > 64:
    effective_window = 8
    print(f"Auto-enabled SHARD_WINDOW={effective_window} for large corpus ({n_shards} shards)")

config = replace(
    config,
    run_id=run_id,
    recompute=bool(RECOMPUTE),
    shard_window=effective_window,
)

summary_rows = [
    ("config", CONFIG_PATH),
    ("run_id", config.run_id),
    ("mode", MODE),
    ("architecture", f"{preset.architecture} ({preset.name})"),
    ("shape", f"d={preset.hidden_size}, L={preset.num_layers}, H={preset.num_heads}, V={preset.vocab_size}"),
    ("corpus_prefix", corpus_pref),
    ("available_shards", n_shards),
    ("shard_window", config.shard_window),
    ("recompute", config.recompute),
    ("micro_batch_size", config.micro_batch_size),
    ("grad_accum_steps", config.grad_accum_steps),
    ("tokens_per_step", config.tokens_per_step),
    ("learning_rate", config.learning_rate),
    ("total_steps", config.total_steps),
]
display(pd.DataFrame(summary_rows, columns=["parameter", "value"]))

data_dest = work_root / "data" / corpus_pref
workdir = work_root / "runs" / config.run_id
workdir.mkdir(parents=True, exist_ok=True)

if config.shard_window > 0:
    corpus, fetcher = fetch_corpus_metadata(
        data_store,
        corpus_pref,
        data_dest,
        eval_tokens=config.eval_sequences * config.seq_len,
    )
else:
    corpus = fetch_tokenized(data_store, corpus_pref, data_dest)
    fetcher = None

splits_summary = {
    split: (
        sum(int(entry.get("tokens", 0)) for entry in info.get("shards", [])),
        len(info.get("shards", [])),
    )
    for split, info in corpus.manifest.get("splits", {}).items()
}
train_tokens = splits_summary.get("train", (0, 0))[0]
epoch_steps = max(1, train_tokens // max(1, config.tokens_per_step))
print("Corpus splits:", ", ".join(f"{k}={t:,} tokens ({s} shards)" for k, (t, s) in splits_summary.items()))
print(f"Approx steps/epoch={epoch_steps:,} | streaming={'on' if fetcher is not None else 'off'}")

In [ ]:
import math
from hamic_jax.lab import run_training_session
from hamic_jax.session import SessionLimits, run_session

if MODE == "bench":
    mb = (
        int(BENCH_MICRO_BATCH)
        if BENCH_MICRO_BATCH > 0
        else (16 if len(devices) == 1 else config.micro_batch_size)
    )
    ga = config.grad_accum_steps
    if 0 < mb < config.micro_batch_size:
        ga = max(1, math.ceil(config.tokens_per_step / (config.seq_len * mb)))
    bench_window = min(config.shard_window, int(BENCH_SHARD_WINDOW)) if config.shard_window > 0 else 0
    bench_lr = float(BENCH_LR) if BENCH_LR is not None else config.learning_rate
    if BENCH_WARMUP is not None:
        bench_total_steps = max(1, math.ceil(config.total_tokens / (config.seq_len * mb * ga)))
        bench_warmup_ratio = min(1.0, max(0.0, float(BENCH_WARMUP) / bench_total_steps))
    else:
        bench_warmup_ratio = config.warmup_ratio

    bench_config = replace(
        config,
        micro_batch_size=mb,
        grad_accum_steps=ga,
        shard_window=bench_window,
        learning_rate=bench_lr,
        warmup_ratio=bench_warmup_ratio,
    )
    limits = SessionLimits(
        steps=int(BENCH_STEPS),
        log_every_steps=1,
        eval_every_steps=int(BENCH_STEPS),
        eval_sequences=min(config.eval_sequences, 32),
        checkpoint_at_end=False,
    )
    result = run_session(
        bench_config,
        preset,
        corpus,
        run_store,
        workdir,
        limits=limits,
        source_commit=installed_commit,
        raw_config=raw_config,
        publish=False,
        log=print,
        handle_preemption=False,
        fetcher=fetcher,
        show_progress=bool(SHOW_PROGRESS),
    )
elif MODE == "train":
    result = run_training_session(
        config,
        preset,
        corpus,
        run_store,
        workdir,
        seconds=float(SESSION_HOURS) * 3600.0,
        source_commit=installed_commit,
        raw_config=raw_config,
        log=print,
        handle_preemption=True,
        fetcher=fetcher,
        show_progress=bool(SHOW_PROGRESS),
        keep_last_checkpoints=int(KEEP_LAST_CHECKPOINTS),
    )
else:
    raise ValueError(f"Unsupported MODE={MODE!r}; expected 'bench' or 'train'.")

tps_values = [float(m["tokens_per_second"]) for m in result.metrics if "tokens_per_second" in m]
steady_tps = sum(tps_values[len(tps_values) // 2 :]) / max(1, len(tps_values) - len(tps_values) // 2) if tps_values else 0.0
hbm_values = [float(m["peak_hbm_gib"]) for m in result.metrics if "peak_hbm_gib" in m]
peak_hbm = max(hbm_values) if hbm_values else 0.0
remaining_steps = max(0, config.total_steps - result.final_step)
hours_left = (remaining_steps * config.tokens_per_step / steady_tps / 3600.0) if steady_tps > 0 else float("inf")

print(
    f"Status={result.status} | final_step={result.final_step} | resumed_from={result.resumed_from_step} "
    f"| published={len(result.published_steps)} | steady_tps={steady_tps:,.0f} tok/s "
    f"| peak_hbm={peak_hbm:.2f} GiB | est_hours_left={hours_left:.2f}"
)
for warning in result.warnings:
    print("WARNING:", warning)

In [ ]:
import matplotlib.pyplot as plt
from hamic_jax.plots import loss_curve

if result.losses:
    fig = loss_curve(result, title=f"{config.run_id} ({MODE})")
    plt.show()
else:
    print("No training loss points recorded.")

if result.evaluations:
    eval_rows = [
        {
            "step": step,
            "val_loss": val_loss,
            "bpb": result.eval_bits_per_byte.get(step),
        }
        for step, val_loss in sorted(result.evaluations.items())
    ]
    display(pd.DataFrame(eval_rows))
else:
    print("No validation evaluations recorded.")